# IOAI — 2025 Team Selection Day3 Nlp Code Difficulty (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day3-nlp-code-difficulty'
for f in ['train.csv','val.csv','test.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','val.csv','test.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
# 의존성: lightgbm (Jupyter/Colab 환경에 없으면 설치)
try:
    import lightgbm  # noqa: F401
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm  # noqa: F401

# Day 3 — Code Difficulty · 모범답안 (Reference Solution)

> **Kazakhstan – Team Selection 2025 · Day 3**

Faithful reproduction of the strong approach (Batyr Yerdenov's TST solution): **hand-crafted code features** (length, token/line counts, loops, conditionals, functions, recursion / DP / hashing keywords) **concatenated with TF-IDF** (word + char n-grams), **random oversampling** to balance classes, and a **LightGBM** classifier. Saves `submission_val.csv` (weighted-F1 grading) and `submission.csv` (Kaggle).

In [ ]:
import os, pandas as pd, numpy as np
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day3_NLP_Code_Difficulty"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."
DATA=_find("train.csv")
train=pd.read_csv(os.path.join(DATA,"train.csv"))   # id, difficulty, code
val  =pd.read_csv(os.path.join(DATA,"val.csv"))     # id, code   (held-out, graded by weighted F1)
test =pd.read_csv(os.path.join(DATA,"test.csv"))    # id, code   (Kaggle test)
print("train",train.shape,"val",val.shape,"test",test.shape,"| classes",sorted(train.difficulty.unique()))


## Hand-crafted code features

In [ ]:
import re, scipy.sparse as sp
KW=["for","while","if","else","def","return","recursion","dp","dynamic","hash","map","set","sort","class","import","try"]
def feats(code):
    c=str(code); lines=c.splitlines() or [""]
    base=[len(c), len(c.split()), len(lines), np.mean([len(l) for l in lines]),
          c.count("("), c.count("{"), c.count("["), c.count("=")]
    low=c.lower(); kw=[low.count(k) for k in KW]
    return base+kw
def feat_matrix(df): return np.array([feats(x) for x in df["code"]], dtype=np.float32)
Ftr, Fva, Fte = feat_matrix(train), feat_matrix(val), feat_matrix(test)


## TF-IDF (word + char) ⊕ features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
wv=TfidfVectorizer(max_features=30000, ngram_range=(1,2), sublinear_tf=True)
cv=TfidfVectorizer(max_features=20000, ngram_range=(3,5), analyzer="char_wb", sublinear_tf=True)
Wtr=wv.fit_transform(train["code"].astype(str)); Ctr=cv.fit_transform(train["code"].astype(str))
def stack(W,C,Fm): return sp.hstack([W, C, sp.csr_matrix(Fm)]).tocsr()
Xtr=stack(Wtr,Ctr,Ftr)
Xva=stack(wv.transform(val["code"].astype(str)),  cv.transform(val["code"].astype(str)),  Fva)
Xte=stack(wv.transform(test["code"].astype(str)), cv.transform(test["code"].astype(str)), Fte)


## Random oversampling + LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.utils import resample
lab=sorted(train["difficulty"].unique()); l2i={l:i for i,l in enumerate(lab)}
ytr=train["difficulty"].map(l2i).values
# random oversample minority classes up to the majority count
idx=np.arange(Xtr.shape[0]); mx=np.bincount(ytr).max(); keep=[]
for c in range(len(lab)):
    ci=idx[ytr==c]; keep.append(resample(ci, replace=True, n_samples=mx, random_state=0))
keep=np.concatenate(keep)
Xb, yb = Xtr[keep], ytr[keep]
clf=lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8,
                       colsample_bytree=0.6, random_state=0)
clf.fit(Xb, yb)
i2l={i:l for l,i in l2i.items()}
def pred(X): return [i2l[i] for i in clf.predict(X)]
pd.DataFrame({"id":val["id"],  "difficulty":pred(Xva)}).to_csv("submission_val.csv",index=False)
pd.DataFrame({"id":test["id"], "difficulty":pred(Xte)}).to_csv("submission.csv",index=False)
print("wrote submission_val.csv / submission.csv")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)